In [33]:
import numpy as np
from numpy.random import default_rng
import scipy
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import datasets
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from scipy.spatial.distance import squareform

import sys
sys.path.insert(0, "../utils")

from embedscore import qm
from embedscore import viz

In [2]:
np.random.seed(42)

In [3]:
sh_points, sh_color = datasets.make_swiss_roll(n_samples=1500, random_state=42, hole=True, noise=0.05)

hd_data = sh_points
labels = sh_color

In [4]:
## t-SNE
tsne_emb = TSNE(n_components=2, random_state=42).fit_transform(hd_data)
pca_emb = PCA(n_components=2).fit_transform(hd_data)


In [ ]:
D_hd = scipy.spatial.distance.squareform(X=scipy.spatial.distance.pdist(X=hd_data, metric='euclidean'), force='tomatrix')
D_ld = scipy.spatial.distance.squareform(X=scipy.spatial.distance.pdist(X=tsne_emb, metric='euclidean'), force='tomatrix')

In [20]:

K = 30 

r_hd = np.argsort(D_hd, axis=1)
r_ld = np.argsort(D_ld, axis=1)

R_hd = np.argsort(r_hd, axis=1)
R_ld = np.argsort(r_ld, axis=1)

In [25]:

def plot_embedding_quality(embedding, labels, link_quality, node_quality, symmetric=False, subsample=False, emb_name=None, crit_name=None, threshold=0.0, point_size=6, point_alpha=0.8,quantiles=False):
    embedding = np.array(embedding)
    fig, ax = plt.subplots(figsize=(26,7), nrows=1, ncols=4)

    '''if emb_name is not None and crit_name is not None:
        fig.suptitle(f'{emb_name} embedding quality metrics - {crit_name}', fontsize=16)'''
    
    ax[0] = viz.visualize_links(embedding,
                    links = link_quality,
                    threshold = threshold,
                    symmetric = symmetric,
                    subsample_edges = subsample,
                    max_edges = 50000,
                    quantiles=quantiles,
                    metric_name = crit_name, 
                    axes = ax[0],
                    point_size = 4)
    ax[1] = viz.visualize_nodes(embedding,                    
                    node_quality,
                    axes=ax[1],
                    metric_name=crit_name,
                    point_size=point_size,
                    point_alpha=point_alpha)
   
    ax[2].scatter(embedding[:, 0], embedding[:, 1], c=labels, s=10, alpha=0.8)
    ax[2].set_title(f'2D {emb_name} embedding')
    ax[2].set_aspect('equal')
    ax[2].legend(fontsize=10, bbox_to_anchor=(1.1, 1.05), markerscale=12)

    ax[3] = fig.add_subplot(1, 4, 4, projection='3d')
    ax[3].scatter(sh_points[:, 0], sh_points[:, 1], sh_points[:, 2], c=labels, s=50, alpha=0.8)
    ax[3].view_init(azim=-66, elev=12)


In [108]:
def random_triplet_accuracy(D_hd: np.ndarray, D_ld: np.ndarray, num_triplets: int = 5):
    
    # Sampling Triplets
    anchors = np.arange(D_hd.shape[0])
    rng = default_rng()
    triplets = rng.choice(anchors, (D_hd.shape[0], num_triplets, 2))
    anchors = anchors.reshape((-1, 1, 1))

    # Generate labels for HD distances
    b = np.broadcast(anchors, triplets)
    distances_hd = np.empty(b.shape)
    distances_hd.flat = [D_hd[u, v] for (u, v) in b]
    labels_hd = distances_hd[:, :, 0] < distances_hd[:, :, 1]

    # Generate labels for LD distances
    distances_ld = np.empty(b.shape)
    distances_ld.flat = [D_ld[u, v] for (u, v) in b]
    labels_ld = distances_ld[:, :, 0] < distances_ld[:, :, 1]

    correct = np.sum(labels_ld == labels_hd, axis=1)
    acc = correct / num_triplets

    return acc

In [109]:
acc = random_triplet_accuracy(D_hd, D_ld, num_triplets=5)

In [110]:
type(acc)

numpy.ndarray